# Module 5: Model-Based RL and Planning

**Spinning Up in Active Inference** | Part 2: The RL Track

---

## Historical Context

In 1948, Edward Tolman published *Cognitive Maps in Rats and Men*, arguing that animals don't just learn stimulus-response associations—they build **internal models** of the world. A rat in a maze doesn't just memorize "turn left, turn right." It constructs a map of the environment and can use it to plan novel routes.

This insight was largely ignored by the behaviorist mainstream for decades. But in 1991, Rich Sutton formalized the idea in the **Dyna architecture**, showing how an RL agent could learn a model of its environment and use it to simulate experience, dramatically accelerating learning.

The distinction between **model-free** and **model-based** methods is one of the most important in all of RL—and it is precisely this distinction that connects RL to Active Inference, where the generative model is the starting point, not a learned add-on.

### What You'll Learn

1. Model-free vs model-based RL: learning value directly vs learning a world model
2. The Dyna architecture: augmenting real experience with simulated experience
3. Model-based value estimation via learned transition and reward models
4. Planning as inference (Botvinick & Toussaint, 2012)
5. The deep connection between RL world models and AIF generative models

## 5.1 Model-Free vs Model-Based: Two Philosophies

Consider two ways to navigate a city:

- **Model-free**: You've walked every route many times. You don't know *why* you turn left at the bakery, but you know it leads to good coffee. If the bakery closes, you're lost.

- **Model-based**: You have a mental map. You know the spatial layout. If the bakery closes, you can immediately plan a detour.

Formally:

| | Model-Free | Model-Based |
|---|---|---|
| **What is learned** | $V(s)$ or $Q(s,a)$ directly | $T(s'|s,a)$ and $R(s,a)$ |
| **How actions are chosen** | $\arg\max_a Q(s,a)$ | Plan using the model |
| **Sample efficiency** | Low (needs many interactions) | High (can simulate) |
| **Computational cost** | Low (just lookup) | High (planning is expensive) |
| **Adaptability** | Low (must re-learn) | High (re-plan with same model) |

The **Dyna architecture** (Sutton, 1991) combines both: learn from real experience AND use the learned model to generate simulated experience for additional learning.

## 5.2 The Dyna Architecture

Dyna's elegant insight: after each real interaction with the environment, take $k$ additional learning steps using **simulated** experience from the learned model.

```
for each real step:
    1. Take action a in state s, observe s', r
    2. Update Q(s,a) from real experience        [model-free update]
    3. Update model: T(s'|s,a), R(s,a)            [model learning]
    4. for k planning steps:                       [model-based planning]
         Sample s_sim, a_sim from previous experience
         Simulate s'_sim, r_sim from model
         Update Q(s_sim, a_sim)                    [simulated update]
```

The parameter $k$ controls the **planning budget**: more planning steps means faster learning but more computation per real step.

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')

from neuronav.agents.td_agents import TDQ
from neuronav.agents.dyna_agents import DynaQ, DynaSR
from neuronav.agents.mb_agents import MBV
from neuronav.envs.grid_env import GridEnv, GridSize, GridObservation
from neuronav.envs.grid_templates import GridTemplate

from utils.plotting import (plot_learning_curve, plot_matrix_heatmap,
                            plot_value_heatmap, dual_perspective_box, COLORS)

np.random.seed(42)
print("Module 5: Model-Based RL and Planning")
print("="*45)

## 5.3 Experiment: DynaQ vs TDQ on FourRooms

The FourRooms environment (Sutton, Precup & Singh, 1999) is a classic benchmark for model-based methods. The environment has narrow doorways connecting four rooms, which means that random exploration takes a long time to discover reward. A model-based agent can "replay" successful doorway crossings to learn much faster.

Let's compare:
- **TDQ** (model-free Q-learning): learns only from real experience
- **DynaQ** (Dyna-Q with $k=10$ planning steps): learns from real + simulated experience

In [ ]:
def run_agent_on_fourrooms(agent, env, n_episodes=200, max_steps=500):
    """Run an agent on FourRooms, tracking cumulative reward per episode."""
    rewards_per_episode = []
    for episode in range(n_episodes):
        obs = env.reset()
        total_reward = 0
        for step in range(max_steps):
            action = agent.sample_action(obs)
            next_obs, reward, done, info = env.step(action)
            agent.update((obs, action, next_obs, reward, done))
            total_reward += reward
            obs = next_obs
            if done:
                break
        rewards_per_episode.append(total_reward)
    return rewards_per_episode


# Create environment
env = GridEnv(template=GridTemplate.four_rooms,
              obs_type=GridObservation.index,
              size=GridSize.small)
state_size = env.observation_space.n
action_size = env.action_space.n

# Create agents
tdq_agent = TDQ(state_size, action_size, lr=0.1, gamma=0.99,
                poltype="softmax", beta=5.0)
dynaq_agent = DynaQ(state_size, action_size, lr=0.1, gamma=0.99,
                    poltype="softmax", beta=5.0)

n_episodes = 200

# Run both agents
print("Training TDQ (model-free)...")
tdq_rewards = run_agent_on_fourrooms(tdq_agent, env, n_episodes)

# Reset environment seed for fair comparison
np.random.seed(42)
print("Training DynaQ (model-based, k=10)...")
dynaq_rewards = run_agent_on_fourrooms(dynaq_agent, env, n_episodes)

print(f"\nTDQ mean reward (last 50 eps):  {np.mean(tdq_rewards[-50:]):.2f}")
print(f"DynaQ mean reward (last 50 eps): {np.mean(dynaq_rewards[-50:]):.2f}")

In [ ]:
# Plot learning curves
plot_learning_curve(
    {"TDQ (model-free)": tdq_rewards,
     "DynaQ (model-based, k=10)": dynaq_rewards},
    xlabel="Episode",
    ylabel="Cumulative Reward",
    title="Sample Efficiency: Model-Free vs Model-Based on FourRooms",
    window=15
)
plt.tight_layout()
plt.show()

print("\nKey observation: DynaQ reaches good performance much earlier.")
print("Each real step generates 10 simulated steps, multiplying the learning signal.")

## 5.4 Inside DynaQ: Visualizing the Internal Model

The key innovation in Dyna is that the agent learns an internal **transition model** $\hat{T}(s'|s,a)$ alongside its Q-values. This model captures the agent's learned understanding of environment dynamics.

Let's peek inside DynaQ's head and see what its internal model looks like compared to the true transition structure.

In [ ]:
# Visualize DynaQ's learned transition model
# The DynaQ agent stores a model of transitions it has experienced

# Extract the internal model — DynaQ stores (state, action) -> (next_state, reward)
# Let's visualize which transitions the agent has learned about
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Build a transition count matrix from the DynaQ model
# The model stores experienced transitions for replay
transition_counts = np.zeros((state_size, state_size))

# Reconstruct from the agent's model storage
if hasattr(dynaq_agent, 'model'):
    for (s, a), (s_prime, r) in dynaq_agent.model.items():
        transition_counts[s, s_prime] += 1
elif hasattr(dynaq_agent, 'history'):
    for (s, a, s_prime, r) in dynaq_agent.history:
        transition_counts[s, s_prime] += 1

# Normalize rows to get transition probabilities
row_sums = transition_counts.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
learned_T = transition_counts / row_sums

# Show a subset of the transition matrix (first 15 states for visibility)
n_show = min(15, state_size)
axes[0].imshow(learned_T[:n_show, :n_show], cmap='Blues', interpolation='nearest')
axes[0].set_title("DynaQ's Learned Transition Model\n(first {} states)".format(n_show))
axes[0].set_xlabel("Next State")
axes[0].set_ylabel("Current State")
plt.colorbar(axes[0].images[0], ax=axes[0], shrink=0.8)

# Show the learned Q-values as a heatmap
q_values = dynaq_agent.q.copy() if hasattr(dynaq_agent, 'q') else np.zeros((state_size, action_size))
v_values = np.max(q_values, axis=1)
grid_side = int(np.sqrt(state_size))

if grid_side**2 == state_size:
    plot_value_heatmap(v_values, grid_side,
                       title="DynaQ Value Function (FourRooms)",
                       ax=axes[1], show_values=False)
else:
    axes[1].bar(range(state_size), v_values, color=COLORS['rl'], alpha=0.7)
    axes[1].set_title("DynaQ Value Function")
    axes[1].set_xlabel("State")
    axes[1].set_ylabel("V(s) = max_a Q(s,a)")

plt.tight_layout()
plt.show()

## 5.5 Model-Based Value Estimation: The MBV Agent

The **MBV** (Model-Based Value) agent in neuro-nav takes model-based RL to its logical conclusion. Instead of using the model only for simulated replay (as in Dyna), MBV:

1. **Learns a full transition model** $\hat{T}(s'|s,a)$—a matrix of transition probabilities
2. **Learns reward weights** $w$ such that $R(s) \approx w^\top \phi(s)$
3. **Plans via value iteration** over the learned model

This is the purest form of model-based RL: the agent has an explicit, inspectable world model.

### The Connection to Active Inference

MBV's transition matrix $\hat{T}(s'|s,a)$ is *exactly* the **B-matrix** in Active Inference. The agent's learned model of how actions change the world IS the generative model's transition dynamics. The key difference: in RL, this model is *learned*; in AIF, it is *specified* as part of the generative model.

In [ ]:
# Create and train an MBV agent
np.random.seed(42)
env = GridEnv(template=GridTemplate.four_rooms,
              obs_type=GridObservation.index,
              size=GridSize.small)
state_size = env.observation_space.n
action_size = env.action_space.n

mbv_agent = MBV(state_size, action_size, lr=0.1, gamma=0.99)

# Train the MBV agent
mbv_rewards = []
for episode in range(200):
    obs = env.reset()
    total_reward = 0
    for step in range(500):
        action = mbv_agent.sample_action(obs)
        next_obs, reward, done, info = env.step(action)
        mbv_agent.update((obs, action, next_obs, reward, done))
        total_reward += reward
        obs = next_obs
        if done:
            break
    mbv_rewards.append(total_reward)

print(f"MBV mean reward (last 50 eps): {np.mean(mbv_rewards[-50:]):.2f}")

In [ ]:
# Visualize the MBV agent's learned transition model
# MBV has an explicit T matrix: T[s', s, a] = P(s'|s,a)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

action_names = ['Up', 'Down', 'Left', 'Right']
n_show = min(15, state_size)

# Show learned T for three actions
for idx, a in enumerate([0, 1, 3]):  # Up, Down, Right
    if hasattr(mbv_agent, 'T'):
        T_a = mbv_agent.T[:n_show, :n_show, a]
    else:
        T_a = np.eye(n_show)  # Fallback

    im = axes[idx].imshow(T_a, cmap='Blues', interpolation='nearest',
                          vmin=0, vmax=1)
    axes[idx].set_title(f"T(s'|s, {action_names[a]})\nLearned by MBV")
    axes[idx].set_xlabel("Current State s")
    axes[idx].set_ylabel("Next State s'")
    plt.colorbar(im, ax=axes[idx], shrink=0.8)

plt.suptitle("MBV Agent's Learned Transition Model", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Each column is a probability distribution: P(s'|s, a).")
print("Compare this to the B-matrix in Active Inference (Module 8) — same semantics!")

## 5.6 The Rosetta Stone: Dyna's T-Table is AIF's B-Matrix

Here we make the connection explicit. Both DynaQ and an Active Inference agent need to represent **transition dynamics**: how does the world change when I act?

| RL (Dyna/MBV) | Active Inference |
|---|---|
| $\hat{T}(s'\|s,a)$ | $B_a = P(s'\|s,a)$ |
| Learned from experience | Specified in generative model |
| Used for simulated replay / planning | Used for belief propagation / EFE |
| Updated via counting or gradient | Updated via Dirichlet learning (Module 11) |

Let's build a small AIF model with an equivalent B-matrix and see they encode the same information.

In [ ]:
# Side-by-side comparison: RL transition model vs AIF B-matrix
# We'll use a small 4-state example for clarity

from alf.generative_model import GenerativeModel

# Define a small environment: 4 states, 2 actions (left/right on a line)
n_states_small = 4
n_actions_small = 2

# RL perspective: learned transition model from experience
# Action 0 = left, Action 1 = right
rl_T = np.zeros((n_states_small, n_states_small, n_actions_small))

# Left action: shift states leftward (with boundary)
for s in range(n_states_small):
    s_left = max(0, s - 1)
    rl_T[s_left, s, 0] = 0.9   # 90% chance of moving left
    rl_T[s, s, 0] = 0.1         # 10% chance of staying

# Right action: shift states rightward (with boundary)
for s in range(n_states_small):
    s_right = min(n_states_small - 1, s + 1)
    rl_T[s_right, s, 1] = 0.9
    rl_T[s, s, 1] = 0.1

# AIF perspective: same dynamics, specified as B-matrix in generative model
aif_B = rl_T.copy()  # Exactly the same matrix!

# Build the AIF generative model
A = [np.eye(n_states_small)]  # Identity observation (fully observable)
B = [aif_B]
C = [np.array([0.0, 0.0, 0.0, 1.0])]  # Prefer state 3 (rightmost)
D = [np.array([1.0, 0.0, 0.0, 0.0])]  # Start in state 0

gm = GenerativeModel(A=A, B=B, C=C, D=D, T=1)
print(f"Generative model: {gm.num_states} states, {gm.num_actions} actions")

# Visualize side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
state_labels = [f"s{i}" for i in range(n_states_small)]

plot_matrix_heatmap(rl_T[:, :, 1], title="RL: Learned T(s'|s, right)",
                    row_labels=state_labels, col_labels=state_labels,
                    cmap='Blues', ax=axes[0])
axes[0].set_xlabel("Current State")
axes[0].set_ylabel("Next State")

plot_matrix_heatmap(gm.B[0][:, :, 1], title="AIF: B-matrix P(s'|s, right)",
                    row_labels=state_labels, col_labels=state_labels,
                    cmap='Oranges', ax=axes[1])
axes[1].set_xlabel("Current State")
axes[1].set_ylabel("Next State")

plt.suptitle("Same Information, Different Notation", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nAre they identical? {np.allclose(rl_T, gm.B[0])}")
print("The RL agent's learned world model IS the AIF agent's B-matrix.")

## 5.7 Planning Depth: How Many Simulated Steps?

In the Dyna architecture, the parameter $k$ controls how many simulated planning steps the agent takes after each real step. More planning means faster learning (per real step) but more computation.

This is directly analogous to the **planning horizon** $T$ in Active Inference: how far into the future does the agent imagine when evaluating policies?

Let's see how different values of $k$ affect learning speed.

In [ ]:
# Experiment: vary number of planning steps in DynaQ
planning_steps_list = [0, 1, 5, 20]
results = {}

for k in planning_steps_list:
    np.random.seed(42)
    env = GridEnv(template=GridTemplate.four_rooms,
                  obs_type=GridObservation.index,
                  size=GridSize.small)
    state_size = env.observation_space.n
    action_size = env.action_space.n

    if k == 0:
        agent = TDQ(state_size, action_size, lr=0.1, gamma=0.99,
                    poltype="softmax", beta=5.0)
        label = f"TDQ (k=0, model-free)"
    else:
        agent = DynaQ(state_size, action_size, lr=0.1, gamma=0.99,
                      poltype="softmax", beta=5.0)
        label = f"DynaQ (k={k})"

    rewards = run_agent_on_fourrooms(agent, env, n_episodes=150)
    results[label] = rewards
    print(f"  {label}: mean reward (last 30) = {np.mean(rewards[-30:]):.2f}")

print("\nDone!")

In [ ]:
# Plot the planning depth comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Learning curves
colors_list = [COLORS['neutral'], '#66BB6A', COLORS['rl'], '#1565C0']
for i, (label, rewards) in enumerate(results.items()):
    vals = np.array(rewards, dtype=float)
    smoothed = np.convolve(vals, np.ones(15)/15, mode='valid')
    axes[0].plot(smoothed, label=label, linewidth=2, color=colors_list[i])

axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Cumulative Reward (smoothed)")
axes[0].set_title("Effect of Planning Depth on Learning")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Bar chart: episodes to reach threshold
threshold = 0.5  # reward threshold
episodes_to_threshold = []
labels = []
for label, rewards in results.items():
    vals = np.array(rewards, dtype=float)
    smoothed = np.convolve(vals, np.ones(15)/15, mode='valid')
    reached = np.where(smoothed > threshold)[0]
    ep = reached[0] if len(reached) > 0 else len(smoothed)
    episodes_to_threshold.append(ep)
    labels.append(label.split('(')[1].rstrip(')'))

axes[1].barh(range(len(labels)), episodes_to_threshold, color=colors_list, alpha=0.85)
axes[1].set_yticks(range(len(labels)))
axes[1].set_yticklabels(labels)
axes[1].set_xlabel(f"Episodes to Reach Reward > {threshold}")
axes[1].set_title("Sample Efficiency vs Planning Depth")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("\nDiminishing returns: going from k=0 to k=5 helps enormously.")
print("Going from k=5 to k=20 helps less — the model has limited accuracy early on.")

## 5.8 Planning as Inference

Botvinick & Toussaint (2012) made a profound observation: **planning can be cast as probabilistic inference** in a graphical model.

The idea: instead of searching over action sequences (as in MCTS or value iteration), define a probabilistic model over trajectories:

$$P(\tau) = P(s_0) \prod_t P(s_{t+1}|s_t, a_t) \cdot P(\mathcal{O}_t = 1 | s_t)$$

where $\mathcal{O}_t$ is an **optimality variable**: $P(\mathcal{O}_t = 1 | s_t) \propto \exp(R(s_t))$. Good states are more likely to be "optimal."

Finding the optimal policy then becomes **posterior inference**: what actions are most likely given that all future states are optimal?

$$\pi^*(a|s) = P(a_t | s_t, \mathcal{O}_{t:T} = 1)$$

This is *exactly* what Active Inference does! The preference distribution $C$ plays the role of the optimality variable, and belief propagation performs the inference.

| Planning as Inference (RL) | Active Inference |
|---|---|
| $P(\mathcal{O}=1 \| s) \propto \exp(R(s))$ | $P(o) = \text{Cat}(C)$ |
| Infer $\pi^*(a\|s, \mathcal{O}=1)$ | Minimize $G(\pi) = -\mathbb{E}[\ln P(o\|C)] - \text{info gain}$ |
| Optimality $\leftrightarrow$ high reward | Optimality $\leftrightarrow$ preferred observations |
| Soft Bellman equation | Free energy minimization |

In [ ]:
# Demonstrate planning as inference on a simple example
# 5-state chain: s0 -- s1 -- s2 -- s3 -- s4 (goal)

n_s = 5
n_a = 2  # 0=left, 1=right

# Transition model (deterministic for simplicity)
T_plan = np.zeros((n_s, n_s, n_a))
for s in range(n_s):
    T_plan[max(0, s-1), s, 0] = 1.0       # left
    T_plan[min(n_s-1, s+1), s, 1] = 1.0   # right

# Reward function: reward at state 4
R = np.array([0.0, 0.0, 0.0, 0.0, 1.0])

# RL approach: value iteration
V_rl = np.zeros(n_s)
gamma = 0.95
for _ in range(100):
    V_new = np.zeros(n_s)
    for s in range(n_s):
        q_values = []
        for a in range(n_a):
            q = sum(T_plan[s_next, s, a] * (R[s] + gamma * V_rl[s_next])
                    for s_next in range(n_s))
            q_values.append(q)
        V_new[s] = max(q_values)
    V_rl = V_new

# AIF approach: preferences encode the same information
# C = ln P(o) where preferred observations correspond to high-reward states
C_aif = np.log(np.array([0.01, 0.01, 0.01, 0.01, 0.96]) + 1e-16)

# Planning as inference: soft value = log(sum_a exp(Q/tau))
# With tau=1, this gives the "soft" optimal value function
V_soft = np.zeros(n_s)
tau = 0.5
for _ in range(100):
    V_new = np.zeros(n_s)
    for s in range(n_s):
        q_values = []
        for a in range(n_a):
            q = sum(T_plan[s_next, s, a] * (R[s] + gamma * V_soft[s_next])
                    for s_next in range(n_s))
            q_values.append(q)
        V_new[s] = tau * np.log(np.sum(np.exp(np.array(q_values) / tau)))
    V_soft = V_new

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].bar(range(n_s), V_rl, color=COLORS['rl'], alpha=0.85)
axes[0].set_title("Hard Value Iteration\n(standard RL)")
axes[0].set_xlabel("State")
axes[0].set_ylabel("V(s)")

axes[1].bar(range(n_s), V_soft, color='#7B1FA2', alpha=0.85)
axes[1].set_title("Soft Value Iteration\n(planning as inference)")
axes[1].set_xlabel("State")
axes[1].set_ylabel("V_soft(s)")

axes[2].bar(range(n_s), C_aif, color=COLORS['aif'], alpha=0.85)
axes[2].set_title("AIF Preferences\nln P(o)")
axes[2].set_xlabel("Observation")
axes[2].set_ylabel("C(o)")

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.suptitle("Three Views of the Same Goal", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Dual Perspective Box
dual_perspective_box(
    rl_text=(
        "Model-based RL agents learn <b>T(s'|s,a)</b> and <b>R(s,a)</b>, then plan "
        "by simulating trajectories through the learned model (Dyna) or solving "
        "the Bellman equation over the model (value iteration). The quality of "
        "planning depends entirely on model accuracy."
    ),
    aif_text=(
        "Active Inference agents start with a <b>generative model P(o,s|a)</b> and "
        "perform inference to plan. The B-matrix IS the transition model; the A-matrix "
        "IS the observation model. These are the <i>same thing</i> viewed differently: "
        "the RL agent's learned model IS the AIF agent's generative model."
    ),
    title="Model-Based RL and Active Inference: Two Sides of the Same Coin"
)

## 5.9 MCTS and Beyond: Tree Search as Approximate Planning

For large state spaces, exact planning (value iteration over the full model) is infeasible. **Monte Carlo Tree Search** (MCTS) provides an approximate alternative:

1. **Select**: Follow the most promising branch (UCB1)
2. **Expand**: Add a new node to the tree
3. **Simulate**: Roll out a random policy from the new node
4. **Backpropagate**: Update values up the tree

MCTS was the key ingredient behind AlphaGo (Silver et al., 2016) and has deep connections to Active Inference:

- MCTS's UCB1 selection balances exploitation (high value) and exploration (uncertainty) — just like EFE's pragmatic and epistemic terms
- MCTS simulates *possible futures* under different action sequences — just like AIF evaluates *policies* by unrolling the generative model
- Both avoid exhaustive planning by focusing computational resources on promising trajectories

## Exercises

### Exercise 5.1: Model Accuracy Matters (Guided)
Add noise to DynaQ's learned model (e.g., randomly corrupt 10% of stored transitions). How does this affect learning compared to a perfect model? What happens with 50% corruption?

*Hint: This connects to "model misspecification" in AIF — what happens when the generative model doesn't match reality?*

### Exercise 5.2: Dyna-SR (Intermediate)
Train a `DynaSR` agent (Dyna with successor representation) on FourRooms. Compare its learning curve to DynaQ and TDQ. When does the SR representation provide advantages?

### Exercise 5.3: Build Your Own Dyna (Open-ended)
Implement a minimal Dyna agent from scratch (without using neuro-nav's DynaQ). You need:
1. A Q-table
2. A model dictionary: `(s, a) -> (s', r)`
3. A planning loop that samples from the model

Verify it matches neuro-nav's DynaQ performance on a simple grid.

---

## Key Takeaways

1. **Model-based RL** learns an internal model $T(s'|s,a)$, $R(s,a)$ and uses it to plan
2. **Dyna** combines model-free learning with model-based replay for improved sample efficiency
3. **The MBV agent's transition matrix is exactly the B-matrix in Active Inference**
4. **Planning as inference** (Botvinick & Toussaint, 2012) casts optimal control as probabilistic inference — the bridge to AIF
5. More planning helps, but with diminishing returns (especially when the model is inaccurate)

## Further Reading

- Sutton, R.S. (1991). Dyna, an integrated architecture for learning, planning, and reacting. *SIGART Bulletin*, 2(4). — The original Dyna paper.
- Botvinick, M. & Toussaint, M. (2012). Planning as inference. *Trends in Cognitive Sciences*, 16(10). — The key bridge paper between RL planning and Bayesian inference.
- Daw, N.D. et al. (2005). Uncertainty-based competition between prefrontal and dorsolateral striatal systems. *PNAS*. — Neuroscience evidence for dual model-free/model-based systems in the brain.
- Silver, D. et al. (2016). Mastering the game of Go with deep neural networks and tree search. *Nature*. — MCTS at scale.
- Parr, T., Pezzulo, G. & Friston, K.J. (2022). *Active Inference*, Chapter 4. — The generative model as the agent's world model.

---

*Next: Module 6 — Policy Gradients and Actor-Critic*